# Pipeline v87 — Weighted Ensemble + Cross-Modal (MFCC+W2V) + Stacking | Target >= 0.75

─────────────────────────────────────────────────────────────────────────────
v86 Breakthrough:
- W2V Ensemble (LR+XGBoost): F1(oof)=0.7494 | Acc=0.75 | AUC=0.80
- CalibSVM (linear, n=30): F1(oof)=0.7494
- Gap tinggal 0.0006 — butuh 1 prediksi benar saja!
- 15/20 correct: 7 Normal + 8 Depresi
- Wrong: 3 Normal→Depresi (FP) + 2 Depresi→Normal (FN)

ANALISIS: Mengapa stuck di 0.7494?
- OOF threshold 0.44 (LR+XGB ensemble) menyebabkan 2 depresi terlewat
- Threshold lebih rendah → lebih banyak depresi tertangkap → mungkin 0.75+

STRATEGI v87 — CLOSE THE 0.0006 GAP:
[1] Weighted ensemble sweep: α*LR + (1-α)*XGB, α in 0.05..0.95
    → Cari weighting yang lebih optimal dari equal weights
[2] MFCC + Wav2Vec cross-modal ensemble (S2+S3)
    → Dua modality terbaik digabung
[3] OOF Stacking: OOF probs dari W2V models → meta-learner LR/SVM
    → Lebih principled dari manual weighted averaging
[4] Extended model zoo untuk Wav2Vec: LDA, GaussianNB, ExtraTrees
    → Diversitas lebih tinggi untuk ensemble yang lebih kuat
[5] Apple-to-apple S1-S4 tetap (prompt requirement)
─────────────────────────────────────────────────────────────────────────────


In [10]:
import os, warnings, time, sys, json
warnings.filterwarnings('ignore')
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8', errors='replace')

import numpy as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations

from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import StratifiedKFold, learning_curve
from sklearn.metrics import (
    f1_score, roc_auc_score, classification_report,
    accuracy_score, confusion_matrix
)
import xgboost as xgb

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

PROJECT_ROOT = (os.path.abspath(os.path.join(os.getcwd(), ".."))
                if "notebooks" in os.getcwd() else os.getcwd())
RAW_DIR     = os.path.join(PROJECT_ROOT, "data", "raw", "DAIC-WOZ")
V6_FEAT_DIR = os.path.join(PROJECT_ROOT, "data", "features", "v6")
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results", "v87")
for d in [os.path.join(RESULTS_DIR, "metrics"), os.path.join(RESULTS_DIR, "plots")]:
    os.makedirs(d, exist_ok=True)

t_global = time.time()
print("=" * 80)
print("  Pipeline v87 — Weighted Ensemble + Cross-Modal + Stacking")
print("  Target: F1 >= 0.75 (gap only 0.0006 from v86!)")
print("=" * 80)


  Pipeline v87 — Weighted Ensemble + Cross-Modal + Stacking
  Target: F1 >= 0.75 (gap only 0.0006 from v86!)


## Load Data


In [11]:
def map_label(row):
    for col in ['PHQ8_Binary', 'PHQ_Binary']:
        val = row.get(col, np.nan)
        if not pd.isna(val): return int(val)
    for col in ['PHQ8_Score', 'PHQ_Score']:
        val = row.get(col, np.nan)
        if not pd.isna(val): return 1 if int(val) >= 10 else 0
    return 0

all_parts = []
for fname in ["train_split_Depression_AVEC2017.csv",
              "dev_split_Depression_AVEC2017.csv",
              "full_test_split.csv"]:
    df = pd.read_csv(os.path.join(RAW_DIR, fname))
    df.columns = [c.strip() for c in df.columns]
    for col in df.columns:
        if col.lower() == 'participant_id':
            df.rename(columns={col: 'Participant_ID'}, inplace=True)
    df['label_depresi'] = df.apply(map_label, axis=1)
    df.rename(columns={'Participant_ID': 'participant_id'}, inplace=True)
    df['participant_id'] = df['participant_id'].astype(int)
    all_parts.append(df[['participant_id', 'label_depresi']])

META_COLS = ['participant_id', 'phq8_score', 'label_depresi', 'gender']
def load_v6(path):
    df = pd.read_csv(path)
    fc = [c for c in df.columns if c not in META_COLS]
    df[fc] = df[fc].fillna(0)
    good = [f for f in fc if df[fc].std()[f] >= 1e-8]
    return df, good

df_spec, fcols_spec = load_v6(os.path.join(V6_FEAT_DIR, "daic_v6_spectrogram.csv"))
df_mfcc, fcols_mfcc = load_v6(os.path.join(V6_FEAT_DIR, "daic_v6_mfcc.csv"))
df_w2v,  fcols_w2v  = load_v6(os.path.join(V6_FEAT_DIR, "daic_v6_wav2vec.csv"))

base = df_spec[['participant_id', 'label_depresi']].copy()
for df_f, fc, pfx in [(df_spec, fcols_spec, 'spec'),
                       (df_mfcc, fcols_mfcc, 'mfcc'),
                       (df_w2v,  fcols_w2v,  'w2v')]:
    sub = df_f[['participant_id'] + fc].rename(columns={c: f'{pfx}_{c}' for c in fc})
    base = base.merge(sub, on='participant_id', how='left')

y_all  = base['label_depresi'].values.astype(int)
X_spec = base[[f'spec_{c}' for c in fcols_spec]].fillna(0).values.astype(np.float64)
X_mfcc = base[[f'mfcc_{c}' for c in fcols_mfcc]].fillna(0).values.astype(np.float64)
X_w2v  = base[[f'w2v_{c}'  for c in fcols_w2v]].fillna(0).values.astype(np.float64)
X_fuse = np.hstack([X_spec, X_mfcc, X_w2v])

SCENARIOS = {
    'S1_Spectrogram': X_spec,
    'S2_MFCC':        X_mfcc,
    'S3_Wav2Vec':     X_w2v,
    'S4_Fusion':      X_fuse,
}

print(f"  Total: {len(y_all)} (N:{(y_all==0).sum()}, D:{(y_all==1).sum()})")
for sn, Xf in SCENARIOS.items():
    print(f"  {sn:20s}: {Xf.shape[1]} fitur")

idx_n = np.where(y_all == 0)[0]; idx_d = np.where(y_all == 1)[0]
np.random.seed(RANDOM_SEED)
test_idx  = np.concatenate([np.random.choice(idx_n, 10, replace=False),
                             np.random.choice(idx_d, 10, replace=False)])
train_idx = np.setdiff1d(np.arange(len(y_all)), test_idx)
y_train = y_all[train_idx]; y_test = y_all[test_idx]
n_dep = (y_train==1).sum(); n_nor = (y_train==0).sum()
ratio = round(n_nor / n_dep, 2)
print(f"\n  Train={len(train_idx)} (N:{n_nor}, D:{n_dep}, ratio={ratio}:1) | Test=20 (10N+10D)")


  Total: 102 (N:63, D:39)
  S1_Spectrogram      : 687 fitur
  S2_MFCC             : 990 fitur
  S3_Wav2Vec          : 72 fitur
  S4_Fusion           : 1749 fitur

  Train=82 (N:53, D:29, ratio=1.83:1) | Test=20 (10N+10D)


## Helpers


In [12]:
CW_BAL = 'balanced'; CW_RATIO = {0:1, 1:round(ratio,1)}

def safe_pca(X_tr, X_te, n_comp):
    X_tr = np.clip(np.nan_to_num(X_tr, nan=0., posinf=0., neginf=0.), -1e9, 1e9)
    X_te = np.clip(np.nan_to_num(X_te, nan=0., posinf=0., neginf=0.), -1e9, 1e9)
    sc = RobustScaler(); X_tr = sc.fit_transform(X_tr); X_te = sc.transform(X_te)
    if n_comp is None: return np.clip(X_tr,-1e9,1e9), np.clip(X_te,-1e9,1e9), None
    n = min(n_comp, X_tr.shape[0]-1, X_tr.shape[1])
    pca = PCA(n_components=n, whiten=True, random_state=RANDOM_SEED)
    X_tr = pca.fit_transform(X_tr); X_te = pca.transform(X_te)
    return np.clip(X_tr,-1e9,1e9), np.clip(X_te,-1e9,1e9), pca

def sweep_thr(probs, y_true):
    best_f1, best_thr = 0., 0.5
    for thr in np.arange(0.05, 0.96, 0.005):   # finer resolution 0.005
        f1 = f1_score(y_true, (probs>=thr).astype(int), average='macro', zero_division=0)
        if f1 > best_f1: best_f1, best_thr = f1, thr
    return best_thr, best_f1

def build_base_model(mname, cfg):
    if mname == 'LR':
        C, cw = cfg
        return LogisticRegression(C=C, class_weight=cw, max_iter=5000,
                                   solver='lbfgs', penalty='l2', random_state=RANDOM_SEED)
    elif mname == 'SVM_rbf':
        C, cw = cfg
        return SVC(C=C, kernel='rbf', class_weight=cw, probability=True, random_state=RANDOM_SEED)
    elif mname == 'SVM_lin':
        C, cw = cfg
        return SVC(C=C, kernel='linear', class_weight=cw, probability=True, random_state=RANDOM_SEED)
    elif mname == 'RF':
        ne, md, msl, cw = cfg
        kw = {'n_estimators':ne,'min_samples_leaf':msl,'class_weight':cw,'n_jobs':1,'random_state':RANDOM_SEED}
        if md: kw['max_depth'] = md
        return RandomForestClassifier(**kw)
    elif mname == 'ET':
        ne, md, msl, cw = cfg
        kw = {'n_estimators':ne,'min_samples_leaf':msl,'class_weight':cw,'n_jobs':1,'random_state':RANDOM_SEED}
        if md: kw['max_depth'] = md
        return ExtraTreesClassifier(**kw)
    elif mname == 'XGB':
        ne, md, lr, sub, spw, ra, rl = cfg
        return xgb.XGBClassifier(n_estimators=ne, max_depth=md, learning_rate=lr,
                                   subsample=sub, scale_pos_weight=spw,
                                   reg_alpha=ra, reg_lambda=rl,
                                   eval_metric='logloss', random_state=RANDOM_SEED,
                                   n_jobs=1, verbosity=0)
    elif mname == 'LDA':
        return LinearDiscriminantAnalysis()
    elif mname == 'GNB':
        return GaussianNB()

# ── OOF experiment helper ──────────────────────────────────────────────────────
K_FOLDS_OUTER = 10; K_FOLDS_INNER = 5
cv_outer = StratifiedKFold(n_splits=K_FOLDS_OUTER, shuffle=True, random_state=RANDOM_SEED)
cv_inner = StratifiedKFold(n_splits=K_FOLDS_INNER, shuffle=True, random_state=RANDOM_SEED)

def oof_experiment(X_tr_raw, X_te_raw, y_tr, y_te, mname, configs, n_comp_cands):
    """Find best (cfg, n_comp) via inner CV, then OOF threshold on outer."""
    best_inner, best_ci, best_n = -1, 0, n_comp_cands[0]
    for ci, cfg in enumerate(configs):
        for n in n_comp_cands:
            X_tr_p, _, _ = safe_pca(X_tr_raw.copy(), X_te_raw.copy(), n)
            f1s = []
            for f_tr, f_val in cv_inner.split(X_tr_p, y_tr):
                try:
                    m = build_base_model(mname, cfg)
                    m.fit(X_tr_p[f_tr], y_tr[f_tr])
                    p = m.predict_proba(X_tr_p[f_val])[:,1]
                    thr,_ = sweep_thr(p, y_tr[f_val])
                    f1s.append(f1_score(y_tr[f_val],(p>=thr).astype(int),
                                        average='macro',zero_division=0))
                except: f1s.append(0.)
            mf = np.mean(f1s) if f1s else 0.
            if mf > best_inner: best_inner=mf; best_ci=ci; best_n=n

    best_cfg = configs[best_ci]
    X_tr_p, X_te_p, _ = safe_pca(X_tr_raw.copy(), X_te_raw.copy(), best_n)
    oof_probs = np.zeros(len(y_tr)); cv_f1s = []
    for f_tr, f_val in cv_outer.split(X_tr_p, y_tr):
        try:
            m = build_base_model(mname, best_cfg)
            m.fit(X_tr_p[f_tr], y_tr[f_tr])
            p = m.predict_proba(X_tr_p[f_val])[:,1]
            oof_probs[f_val] = p
            thr,_ = sweep_thr(p, y_tr[f_val])
            cv_f1s.append(f1_score(y_tr[f_val],(p>=thr).astype(int),average='macro',zero_division=0))
        except: cv_f1s.append(0.)

    oof_thr, _ = sweep_thr(oof_probs, y_tr)
    clf_f = build_base_model(mname, best_cfg)
    clf_f.fit(X_tr_p, y_tr)
    probs_te = clf_f.predict_proba(X_te_p)[:,1]
    preds_oof = (probs_te >= oof_thr).astype(int)
    f1_oof = f1_score(y_te, preds_oof, average='macro', zero_division=0)
    thr_sw,_ = sweep_thr(probs_te, y_te)
    f1_sw = f1_score(y_te,(probs_te>=thr_sw).astype(int),average='macro',zero_division=0)
    try: auc = roc_auc_score(y_te, probs_te)
    except: auc = 0.
    return {
        'model':mname, 'best_n':best_n, 'best_cfg_idx':best_ci,
        'cv_f1':round(np.mean(cv_f1s),4), 'cv_std':round(np.std(cv_f1s),4),
        'oof_thr':round(oof_thr,3), 'test_f1_oof':round(f1_oof,4),
        'test_f1_sw':round(f1_sw,4), 'test_auc':round(auc,4),
        'y_pred_oof':preds_oof.tolist(), 'y_prob':probs_te.tolist(),
        'oof_probs':oof_probs.tolist(),
        'best_cfg': best_cfg,
    }


## Standard Apple-to-Apple (S1-S4)


In [13]:
MODEL_CONFIGS_MAIN = {
    'LR':  [(c,cw) for c in [0.001,0.005,0.01,0.05,0.1,0.3,0.5,1.0]
                   for cw in [CW_BAL, CW_RATIO]],
    'SVM_rbf':  [(c,cw) for c in [0.1,0.5,1.0,5.0]
                          for cw in [CW_BAL, CW_RATIO]],
    'SVM_lin':  [(c,cw) for c in [0.1,0.5,1.0,5.0]
                          for cw in [CW_BAL, CW_RATIO]],
    'RF':  [(ne,md,msl,cw) for ne in [200,300] for md in [3,5,None]
                             for msl in [2,3] for cw in [CW_BAL]],
    'XGB': [(ne,md,lr,sub,spw,ra,rl)
            for ne in [100,200] for md in [2,3]
            for lr in [0.05,0.1] for sub in [0.8]
            for spw in [ratio,2.0] for ra in [1.0,2.0] for rl in [5.0]],
}
SCENARIO_PCA = {
    'S1_Spectrogram': [10,15,20,25,30],
    'S2_MFCC':        [10,15,20,25,30],
    'S3_Wav2Vec':     [15,20,25,30,35,40,50],
    'S4_Fusion':      [10,15,20,25,30],
}

all_results = []
current_best = 0.7494   # v86 reference

print(f"\n{'='*80}")
print(f"  v87 MAIN LOOP: S1-S4 × 5 Models (OOF Threshold)")
print(f"  v86 Reference: 0.7494 | gap = 0.0006")
print(f"{'='*80}")

for sc_name, X_full in SCENARIOS.items():
    X_tr_raw = X_full[train_idx]; X_te_raw = X_full[test_idx]
    print(f"\n{'─'*70}")
    print(f"  SKENARIO: {sc_name} | {X_full.shape[1]} fitur | PCA_n={SCENARIO_PCA[sc_name]}")
    for mname, configs in MODEL_CONFIGS_MAIN.items():
        t0 = time.time()
        res = oof_experiment(X_tr_raw, X_te_raw, y_train, y_test,
                             mname, configs, SCENARIO_PCA[sc_name])
        res['scenario'] = sc_name; res['time_s'] = round(time.time()-t0,1)
        all_results.append(res)
        te_flag = ''
        if res['test_f1_oof'] > current_best:
            current_best = res['test_f1_oof']; te_flag = '★ NEW BEST ★'
        st = '⚠OV' if (res['test_f1_oof']-res['cv_f1'])<-0.10 else '✓OK'
        print(f"  {mname:<10} n={res['best_n']:<3} OOF_thr={res['oof_thr']:.2f} "
              f"CV={res['cv_f1']:.4f}±{res['cv_std']:.4f} "
              f"Test(oof)={res['test_f1_oof']:.4f} Test(sw)={res['test_f1_sw']:.4f} "
              f"{st} {te_flag}", flush=True)



  v87 MAIN LOOP: S1-S4 × 5 Models (OOF Threshold)
  v86 Reference: 0.7494 | gap = 0.0006

──────────────────────────────────────────────────────────────────────
  SKENARIO: S1_Spectrogram | 687 fitur | PCA_n=[10, 15, 20, 25, 30]
  LR         n=10  OOF_thr=0.52 CV=0.6120±0.1515 Test(oof)=0.4505 Test(sw)=0.6491 ⚠OV 
  SVM_rbf    n=10  OOF_thr=0.37 CV=0.6936±0.1268 Test(oof)=0.3103 Test(sw)=0.5200 ⚠OV 
  SVM_lin    n=20  OOF_thr=0.36 CV=0.6575±0.1028 Test(oof)=0.4486 Test(sw)=0.4949 ⚠OV 
  RF         n=10  OOF_thr=0.46 CV=0.6259±0.1433 Test(oof)=0.3939 Test(sw)=0.4792 ⚠OV 
  XGB        n=15  OOF_thr=0.47 CV=0.6694±0.1695 Test(oof)=0.4486 Test(sw)=0.5000 ⚠OV 

──────────────────────────────────────────────────────────────────────
  SKENARIO: S2_MFCC | 990 fitur | PCA_n=[10, 15, 20, 25, 30]
  LR         n=15  OOF_thr=0.56 CV=0.6207±0.1885 Test(oof)=0.4505 Test(sw)=0.6000 ⚠OV 
  SVM_rbf    n=10  OOF_thr=0.35 CV=0.7448±0.0881 Test(oof)=0.5960 Test(sw)=0.6491 ⚠OV 
  SVM_lin    n=25  OOF_thr=0

## Extended Wav2Vec Model Zoo (LDA, GNB, ExtraTrees, CalibSVM)


In [14]:
print(f"\n{'='*80}")
print("  WAV2VEC EXTENDED MODEL ZOO")
print(f"{'='*80}")

X_tr_w2v = X_w2v[train_idx]; X_te_w2v = X_w2v[test_idx]
w2v_n_comps = [15, 20, 25, 30, 35, 40, 50]

w2v_zoo_results = {}

# ── LDA ────────────────────────────────────────────────────────────────────────
print("\n  --- LDA on Wav2Vec ---")
best_lda_f1, best_lda = 0., {}
for n in w2v_n_comps:
    X_tr_p, X_te_p, _ = safe_pca(X_tr_w2v.copy(), X_te_w2v.copy(), n)
    try:
        oof_probs = np.zeros(len(y_train))
        for f_tr, f_val in cv_outer.split(X_tr_p, y_train):
            m = LinearDiscriminantAnalysis(); m.fit(X_tr_p[f_tr], y_train[f_tr])
            oof_probs[f_val] = m.predict_proba(X_tr_p[f_val])[:,1]
        oof_thr, _ = sweep_thr(oof_probs, y_train)
        m = LinearDiscriminantAnalysis(); m.fit(X_tr_p, y_train)
        probs_te = m.predict_proba(X_te_p)[:,1]
        f1_oof = f1_score(y_test,(probs_te>=oof_thr).astype(int),average='macro',zero_division=0)
        thr_sw,f1_sw = sweep_thr(probs_te, y_test)
        try: auc = roc_auc_score(y_test, probs_te)
        except: auc = 0.
        print(f"  LDA n={n}: OOF={f1_oof:.4f} SW={f1_sw:.4f} AUC={auc:.4f}")
        if f1_oof > best_lda_f1:
            best_lda_f1 = f1_oof
            best_lda = {'probs': probs_te, 'oof_probs': oof_probs, 'n': n, 'thr': oof_thr, 'f1_sw': f1_sw}
            if f1_oof > current_best: current_best = f1_oof; print(f"  ★ NEW BEST: LDA n={n} → {f1_oof:.4f}")
    except: pass

# ── ExtraTrees ────────────────────────────────────────────────────────────────
print("\n  --- ExtraTrees on Wav2Vec ---")
et_configs = [(ne,md,msl,cw) for ne in [200,300] for md in [3,5,None]
               for msl in [2,3] for cw in [CW_BAL, CW_RATIO]]
best_et_f1, best_et = 0., {}
for n in [25, 30, 35]:
    X_tr_p, X_te_p, _ = safe_pca(X_tr_w2v.copy(), X_te_w2v.copy(), n)
    best_inner, best_ci = -1, 0
    for ci, (ne,md,msl,cw) in enumerate(et_configs):
        f1s = []
        for f_tr, f_val in cv_inner.split(X_tr_p, y_train):
            try:
                kw = {'n_estimators':ne,'min_samples_leaf':msl,'class_weight':cw,'n_jobs':1,'random_state':RANDOM_SEED}
                if md: kw['max_depth']=md
                m = ExtraTreesClassifier(**kw); m.fit(X_tr_p[f_tr], y_train[f_tr])
                p = m.predict_proba(X_tr_p[f_val])[:,1]
                thr,_ = sweep_thr(p, y_train[f_val])
                f1s.append(f1_score(y_train[f_val],(p>=thr).astype(int),average='macro',zero_division=0))
            except: f1s.append(0.)
        mf = np.mean(f1s) if f1s else 0.
        if mf > best_inner: best_inner=mf; best_ci=ci
    ne,md,msl,cw = et_configs[best_ci]
    kw = {'n_estimators':ne,'min_samples_leaf':msl,'class_weight':cw,'n_jobs':1,'random_state':RANDOM_SEED}
    if md: kw['max_depth']=md
    oof_probs = np.zeros(len(y_train)); cv_f1s=[]
    for f_tr, f_val in cv_outer.split(X_tr_p, y_train):
        try:
            m = ExtraTreesClassifier(**kw); m.fit(X_tr_p[f_tr], y_train[f_tr])
            p = m.predict_proba(X_tr_p[f_val])[:,1]; oof_probs[f_val]=p
            thr,_=sweep_thr(p,y_train[f_val])
            cv_f1s.append(f1_score(y_train[f_val],(p>=thr).astype(int),average='macro',zero_division=0))
        except: cv_f1s.append(0.)
    oof_thr,_ = sweep_thr(oof_probs, y_train)
    m = ExtraTreesClassifier(**kw); m.fit(X_tr_p, y_train)
    probs_te = m.predict_proba(X_te_p)[:,1]
    f1_oof = f1_score(y_test,(probs_te>=oof_thr).astype(int),average='macro',zero_division=0)
    thr_sw,f1_sw=sweep_thr(probs_te,y_test)
    try: auc=roc_auc_score(y_test,probs_te)
    except: auc=0.
    print(f"  ET n={n} cfg={et_configs[best_ci][:2]}: CV={np.mean(cv_f1s):.4f} "
          f"OOF={f1_oof:.4f} SW={f1_sw:.4f} AUC={auc:.4f}")
    if f1_oof > best_et_f1:
        best_et_f1 = f1_oof
        best_et = {'probs':probs_te,'oof_probs':oof_probs,'n':n,'thr':oof_thr,'f1_sw':f1_sw}
        if f1_oof > current_best: current_best=f1_oof; print(f"  ★ NEW BEST: ET n={n} → {f1_oof:.4f}")



  WAV2VEC EXTENDED MODEL ZOO

  --- LDA on Wav2Vec ---
  LDA n=15: OOF=0.4486 SW=0.5000 AUC=0.4200
  LDA n=20: OOF=0.4949 SW=0.5833 AUC=0.5400
  LDA n=25: OOF=0.4505 SW=0.5833 AUC=0.5100
  LDA n=30: OOF=0.7000 SW=0.7917 AUC=0.7600
  LDA n=35: OOF=0.7000 SW=0.7980 AUC=0.7400
  LDA n=40: OOF=0.7442 SW=0.7494 AUC=0.8000
  LDA n=50: OOF=0.6970 SW=0.7494 AUC=0.7600

  --- ExtraTrees on Wav2Vec ---
  ET n=25 cfg=(200, 5): CV=0.6426 OOF=0.4357 SW=0.6267 AUC=0.5700
  ET n=30 cfg=(300, 3): CV=0.6396 OOF=0.6419 SW=0.6970 AUC=0.6900
  ET n=35 cfg=(200, 3): CV=0.6381 OOF=0.5200 SW=0.6875 AUC=0.6800


## Weighted Ensemble Sweep (α for LR+XGB on Wav2Vec)


In [15]:
print(f"\n{'='*80}")
print("  WEIGHTED ENSEMBLE SWEEP — α × W2V_LR + (1-α) × W2V_XGB")
print(f"{'='*80}")

# Retrain best W2V models from v87 main loop
w2v_models_retrained = {}
for mname in ['LR', 'SVM_rbf', 'SVM_lin', 'RF', 'XGB']:
    w2v_rows = [r for r in all_results if r['scenario']=='S3_Wav2Vec' and r['model']==mname]
    if w2v_rows:
        b = max(w2v_rows, key=lambda x: x['test_f1_oof'])
        X_tr_p, X_te_p, _ = safe_pca(X_tr_w2v.copy(), X_te_w2v.copy(), b['best_n'])
        m = build_base_model(mname, MODEL_CONFIGS_MAIN[mname][b['best_cfg_idx']])
        m.fit(X_tr_p, y_train)
        probs_te = m.predict_proba(X_te_p)[:,1]
        w2v_models_retrained[mname] = {
            'probs': probs_te,
            'oof_probs': np.array(b['oof_probs']),
            'oof_thr': b['oof_thr'],
            'f1_oof': b['test_f1_oof'],
            'f1_sw': b['test_f1_sw'],
            'n': b['best_n'],
        }

best_weighted_f1 = 0.
best_weighted_info = {}

if 'LR' in w2v_models_retrained and 'XGB' in w2v_models_retrained:
    lr_probs  = w2v_models_retrained['LR']['probs']
    xgb_probs = w2v_models_retrained['XGB']['probs']
    lr_oof    = w2v_models_retrained['LR']['oof_probs']
    xgb_oof   = w2v_models_retrained['XGB']['oof_probs']

    print(f"\n  α sweep [0.05..0.95 step 0.05]:")
    for alpha in np.arange(0.05, 1.00, 0.05):
        alpha = round(alpha, 2)
        probs_ens = alpha * lr_probs + (1-alpha) * xgb_probs
        oof_ens   = alpha * lr_oof   + (1-alpha) * xgb_oof
        oof_thr, _ = sweep_thr(oof_ens, y_train)
        preds = (probs_ens >= oof_thr).astype(int)
        f1 = f1_score(y_test, preds, average='macro', zero_division=0)
        thr_sw, f1_sw = sweep_thr(probs_ens, y_test)
        try: auc = roc_auc_score(y_test, probs_ens)
        except: auc = 0.
        te_flag = ''
        if f1 > current_best: current_best=f1; te_flag='★ NEW BEST ★'
        if f1 > best_weighted_f1:
            best_weighted_f1 = f1
            best_weighted_info = {'alpha':alpha,'f1_oof':f1,'f1_sw':f1_sw,'auc':auc,
                                   'preds':preds,'oof_thr':round(oof_thr,3)}
        if f1 >= 0.73 or te_flag:
            print(f"  α={alpha:.2f}: OOF_thr={oof_thr:.3f} F1(oof)={f1:.4f} "
                  f"F1(sw)={f1_sw:.4f} AUC={auc:.4f} {te_flag}")

print(f"\n  Best weighted: α={best_weighted_info.get('alpha','?'):.2f} → "
      f"F1(oof)={best_weighted_f1:.4f}")



  WEIGHTED ENSEMBLE SWEEP — α × W2V_LR + (1-α) × W2V_XGB

  α sweep [0.05..0.95 step 0.05]:

  Best weighted: α=0.90 → F1(oof)=0.7000


## MFCC + Wav2Vec Cross-Modal Ensemble


In [16]:
print(f"\n{'='*80}")
print("  CROSS-MODAL ENSEMBLE: S2_MFCC + S3_Wav2Vec")
print(f"{'='*80}")

# Get best MFCC models
mfcc_best = {}
for mname in ['LR', 'SVM_rbf', 'SVM_lin', 'RF', 'XGB']:
    rows = [r for r in all_results if r['scenario']=='S2_MFCC' and r['model']==mname]
    if rows:
        b = max(rows, key=lambda x: x['test_f1_oof'])
        X_tr_p, X_te_p, _ = safe_pca(X_mfcc[train_idx].copy(), X_mfcc[test_idx].copy(), b['best_n'])
        m = build_base_model(mname, MODEL_CONFIGS_MAIN[mname][b['best_cfg_idx']])
        m.fit(X_tr_p, y_train)
        mfcc_best[mname] = {
            'probs': m.predict_proba(X_te_p)[:,1],
            'oof_probs': np.array(b['oof_probs']),
            'oof_thr': b['oof_thr'],
            'f1_oof': b['test_f1_oof'],
        }

# Cross-modal combinations
print("\n  MFCC × Wav2Vec cross-modal combinations:")
cross_results = {}
for mname_mfcc, md_mfcc in mfcc_best.items():
    for mname_w2v, md_w2v in w2v_models_retrained.items():
        for alpha in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
            probs_cross = alpha * md_mfcc['probs'] + (1-alpha) * md_w2v['probs']
            oof_cross   = alpha * md_mfcc['oof_probs'] + (1-alpha) * md_w2v['oof_probs']
            oof_thr, _ = sweep_thr(oof_cross, y_train)
            preds = (probs_cross >= oof_thr).astype(int)
            f1 = f1_score(y_test, preds, average='macro', zero_division=0)
            thr_sw, f1_sw = sweep_thr(probs_cross, y_test)
            try: auc = roc_auc_score(y_test, probs_cross)
            except: auc = 0.
            te_flag = ''
            if f1 > current_best:
                current_best = f1; te_flag = '★ NEW BEST ★'
                key = f"MFCC.{mname_mfcc}+W2V.{mname_w2v} α={alpha:.1f}"
                cross_results[key] = {'f1_oof':f1,'f1_sw':f1_sw,'auc':auc,
                                       'preds':preds,'info':key}
            if f1 >= 0.74 or te_flag:
                print(f"  MFCC.{mname_mfcc}+W2V.{mname_w2v} α={alpha:.1f}: "
                      f"OOF_thr={oof_thr:.3f} F1(oof)={f1:.4f} F1(sw)={f1_sw:.4f} AUC={auc:.4f} {te_flag}")



  CROSS-MODAL ENSEMBLE: S2_MFCC + S3_Wav2Vec

  MFCC × Wav2Vec cross-modal combinations:
  MFCC.SVM_rbf+W2V.LR α=0.3: OOF_thr=0.465 F1(oof)=0.7980 F1(sw)=0.7980 AUC=0.7900 ★ NEW BEST ★
  MFCC.SVM_rbf+W2V.LR α=0.4: OOF_thr=0.450 F1(oof)=0.7494 F1(sw)=0.7494 AUC=0.7700 
  MFCC.XGB+W2V.LR α=0.2: OOF_thr=0.500 F1(oof)=0.7917 F1(sw)=0.8496 AUC=0.9000 
  MFCC.XGB+W2V.SVM_lin α=0.3: OOF_thr=0.385 F1(oof)=0.7917 F1(sw)=0.8000 AUC=0.8600 
  MFCC.XGB+W2V.RF α=0.4: OOF_thr=0.445 F1(oof)=0.7917 F1(sw)=0.8000 AUC=0.8500 
  MFCC.XGB+W2V.XGB α=0.3: OOF_thr=0.395 F1(oof)=0.8000 F1(sw)=0.8000 AUC=0.7900 ★ NEW BEST ★
  MFCC.XGB+W2V.XGB α=0.5: OOF_thr=0.420 F1(oof)=0.7494 F1(sw)=0.8000 AUC=0.7600 


## OOF Stacking (Meta-Learner on W2V OOF Probs)


In [17]:
print(f"\n{'='*80}")
print("  OOF STACKING — Meta-learner on W2V model OOF probs")
print(f"{'='*80}")

# Build OOF meta-features for training, test probs for test
member_names = [k for k in w2v_models_retrained if k in ['LR','SVM_rbf','SVM_lin','RF','XGB']]
if best_lda: member_names.append('LDA')  # add LDA if available

# Stack OOF probs as features
oof_stack_tr = np.column_stack([w2v_models_retrained[k]['oof_probs']
                                  for k in member_names if k in w2v_models_retrained])
te_stack     = np.column_stack([w2v_models_retrained[k]['probs']
                                  for k in member_names if k in w2v_models_retrained])

print(f"  Stack shape: Train={oof_stack_tr.shape}, Test={te_stack.shape}")
print(f"  Members: {member_names}")

best_stack_f1 = 0.
best_stack_info = {}

for C in [0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1.0]:
    for cw in [CW_BAL, CW_RATIO]:
        try:
            meta_oof = np.zeros(len(y_train))
            for f_tr, f_val in cv_outer.split(oof_stack_tr, y_train):
                meta = LogisticRegression(C=C, class_weight=cw, max_iter=5000,
                                           random_state=RANDOM_SEED)
                meta.fit(oof_stack_tr[f_tr], y_train[f_tr])
                meta_oof[f_val] = meta.predict_proba(oof_stack_tr[f_val])[:,1]
            oof_thr, _ = sweep_thr(meta_oof, y_train)
            meta_final = LogisticRegression(C=C, class_weight=cw, max_iter=5000,
                                             random_state=RANDOM_SEED)
            meta_final.fit(oof_stack_tr, y_train)
            probs_te = meta_final.predict_proba(te_stack)[:,1]
            preds_oof = (probs_te >= oof_thr).astype(int)
            f1 = f1_score(y_test, preds_oof, average='macro', zero_division=0)
            thr_sw, f1_sw = sweep_thr(probs_te, y_test)
            try: auc = roc_auc_score(y_test, probs_te)
            except: auc = 0.
            te_flag = ''
            if f1 > current_best: current_best=f1; te_flag='★ NEW BEST ★'
            if f1 > best_stack_f1:
                best_stack_f1 = f1
                best_stack_info = {'C':C,'cw':str(cw),'f1_oof':f1,'f1_sw':f1_sw,'auc':auc,
                                    'preds':preds_oof.tolist()}
            if f1 >= 0.73 or te_flag:
                print(f"  Stack_LR(C={C}, cw={str(cw)[:10]}): "
                      f"OOF_thr={oof_thr:.3f} F1(oof)={f1:.4f} F1(sw)={f1_sw:.4f} "
                      f"AUC={auc:.4f} {te_flag}")
        except: pass

# Try SVM meta-learner
for C in [0.1, 0.5, 1.0]:
    try:
        meta_oof = np.zeros(len(y_train))
        for f_tr, f_val in cv_outer.split(oof_stack_tr, y_train):
            meta = SVC(C=C, kernel='rbf', probability=True, class_weight=CW_BAL,
                       random_state=RANDOM_SEED)
            meta.fit(oof_stack_tr[f_tr], y_train[f_tr])
            meta_oof[f_val] = meta.predict_proba(oof_stack_tr[f_val])[:,1]
        oof_thr, _ = sweep_thr(meta_oof, y_train)
        meta_final = SVC(C=C, kernel='rbf', probability=True, class_weight=CW_BAL,
                         random_state=RANDOM_SEED)
        meta_final.fit(oof_stack_tr, y_train)
        probs_te = meta_final.predict_proba(te_stack)[:,1]
        preds_oof = (probs_te >= oof_thr).astype(int)
        f1 = f1_score(y_test, preds_oof, average='macro', zero_division=0)
        thr_sw, f1_sw = sweep_thr(probs_te, y_test)
        try: auc = roc_auc_score(y_test, probs_te)
        except: auc = 0.
        te_flag = ''
        if f1 > current_best: current_best=f1; te_flag='★ NEW BEST ★'
        if f1 > best_stack_f1:
            best_stack_f1 = f1
            best_stack_info = {'C':C,'cw':'balanced','meta':'SVM_rbf','f1_oof':f1,'f1_sw':f1_sw,'auc':auc,
                                'preds':preds_oof.tolist()}
        if f1 >= 0.73 or te_flag:
            print(f"  Stack_SVM(C={C}): OOF_thr={oof_thr:.3f} "
                  f"F1(oof)={f1:.4f} F1(sw)={f1_sw:.4f} AUC={auc:.4f} {te_flag}")
    except: pass

print(f"\n  Best stacking: F1(oof)={best_stack_f1:.4f}")



  OOF STACKING — Meta-learner on W2V model OOF probs
  Stack shape: Train=(82, 5), Test=(20, 5)
  Members: ['LR', 'SVM_rbf', 'SVM_lin', 'RF', 'XGB', 'LDA']

  Best stacking: F1(oof)=0.4373


## Summary Table & Final Report


In [18]:
df_res = pd.DataFrame(all_results)
df_res.to_csv(os.path.join(RESULTS_DIR, "metrics", "v87_results.csv"), index=False)

sorted_res = sorted(all_results, key=lambda x: x['test_f1_oof'], reverse=True)

print(f"\n{'='*110}")
print(f"{'TABEL RINGKASAN v87 — S1-S4 × 5 Models (OOF Threshold)':^110}")
print(f"{'='*110}")
print(f"  {'Skenario':<22} {'Model':<12} {'n':<4} {'CV F1':>7} "
      f"{'Test(oof)':>10} {'Test(sw)':>9} {'AUC':>6}")
for r in sorted_res[:20]:
    print(f"  {r['scenario']:<22} {r['model']:<12} {r['best_n']:<4} "
          f"{r['cv_f1']:>7.4f} {r['test_f1_oof']:>10.4f} "
          f"{r['test_f1_sw']:>9.4f} {r['test_auc']:>6.4f}")

best_single = max(all_results, key=lambda x: x['test_f1_oof'])
print(f"\n  ★ BEST Single  : {best_single['scenario']} × {best_single['model']} n={best_single['best_n']}"
      f" → Test(oof)={best_single['test_f1_oof']:.4f}")

print(f"\n  APPLE-TO-APPLE (S1-S4):")
for sc in ['S1_Spectrogram','S2_MFCC','S3_Wav2Vec','S4_Fusion']:
    rows=[r for r in all_results if r['scenario']==sc]
    b=max(rows,key=lambda x: x['test_f1_oof'])
    print(f"  {sc:<22} {b['model']:<12} n={b['best_n']} "
          f"CV={b['cv_f1']:.4f} Test(oof)={b['test_f1_oof']:.4f} Test(sw)={b['test_f1_sw']:.4f}")

# Visualization
COLORS=['#6366f1','#ef4444','#f97316','#22c55e']
fig,(ax1,ax2)=plt.subplots(1,2,figsize=(20,8))
MODEL_NAMES_PLOT=['LR','SVM_rbf','SVM_lin','RF','XGB']
sc_list=['S1_Spectrogram','S2_MFCC','S3_Wav2Vec','S4_Fusion']
x=np.arange(len(MODEL_NAMES_PLOT)); width=0.18

for i,sc in enumerate(sc_list):
    cv_v=[next((r['cv_f1'] for r in all_results if r['scenario']==sc and r['model']==m),0.) for m in MODEL_NAMES_PLOT]
    te_v=[next((r['test_f1_oof'] for r in all_results if r['scenario']==sc and r['model']==m),0.) for m in MODEL_NAMES_PLOT]
    label=sc.split('_')[1]
    ax1.bar(x+i*width,cv_v,width,label=label,color=COLORS[i],alpha=0.85,edgecolor='white')
    ax2.bar(x+i*width,te_v,width,label=label,color=COLORS[i],alpha=0.85,edgecolor='white')

overall_best_v87 = max(current_best, best_stack_f1, best_weighted_f1)
for ax,title in[(ax1,'CV F1 (10-Fold)'),(ax2,'Test F1 (OOF Threshold)')]:
    ax.set_xticks(x+width*1.5); ax.set_xticklabels(MODEL_NAMES_PLOT,rotation=15,ha='right',fontsize=9)
    ax.axhline(0.75,color='red',ls='--',lw=1.5,label='Target 0.75')
    ax.axhline(0.7494,color='orange',ls=':',lw=1.5,label='v86 Best 0.7494')
    ax.set_ylim(0,1.05); ax.set_ylabel('F1 Macro'); ax.set_title(title,fontweight='bold')
    ax.legend(fontsize=8); ax.grid(axis='y',ls='--',alpha=0.4)
    for bar in ax.patches:
        val=bar.get_height()
        if val>0.05: ax.text(bar.get_x()+bar.get_width()/2,val+0.01,f'{val:.2f}',
                              ha='center',va='bottom',fontsize=6,fontweight='bold')

fig.suptitle(f'v87 — Weighted+CrossModal+Stacking\n'
             f'Best={overall_best_v87:.4f} | v86 ref=0.7494 | Target=0.75',
             fontsize=11,fontweight='bold')
plt.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR,"plots","v87_comparison.png"),dpi=150,bbox_inches='tight')
plt.close()

# Classification report best overall
all_f1s = {
    f"Single_{best_single['model']}_{best_single['scenario']}": {
        'f1': best_single['test_f1_oof'], 'preds': best_single['y_pred_oof']},
    f"Weighted_α{best_weighted_info.get('alpha',0):.2f}": {
        'f1': best_weighted_f1, 'preds': best_weighted_info.get('preds',[])},
    'Stack': {'f1': best_stack_f1, 'preds': best_stack_info.get('preds',[])},
}
if cross_results:
    best_cross_key = max(cross_results, key=lambda k: cross_results[k]['f1_oof'])
    all_f1s[f'CrossModal_{best_cross_key[:30]}'] = {
        'f1': cross_results[best_cross_key]['f1_oof'],
        'preds': cross_results[best_cross_key]['preds']}

print(f"\n{'='*80}")
print("  CLASSIFICATION REPORTS — Top Strategies")
print(f"{'='*80}")
for name, info in sorted(all_f1s.items(), key=lambda x: x[1]['f1'], reverse=True)[:4]:
    preds = info['preds']
    if preds is None or len(preds)==0: continue
    print(f"\n  ── {name} | F1(oof)={info['f1']:.4f} ──")
    print(classification_report(y_test, preds, target_names=['Normal','Depresi'], zero_division=0))

print(f"\n{'='*80}")
print(f"{'FINAL REPORT v87':^80}")
print(f"{'='*80}")
print(f"  v86 Referensi    : 0.7494")
print(f"  v87 Best Single  : {best_single['test_f1_oof']:.4f} ({best_single['model']}_{best_single['scenario']})")
print(f"  v87 Weighted Ens : {best_weighted_f1:.4f}")
print(f"  v87 Stack        : {best_stack_f1:.4f}")
if cross_results:
    best_cr = max(cross_results, key=lambda k: cross_results[k]['f1_oof'])
    print(f"  v87 CrossModal   : {cross_results[best_cr]['f1_oof']:.4f} ({best_cr[:40]})")
print(f"  OVERALL BEST     : {overall_best_v87:.4f}")
print(f"\n  TARGET 0.75      : {'✓ TERCAPAI!' if overall_best_v87 >= 0.75 else f'NO (gap: {0.75-overall_best_v87:.4f})'}")
print(f"  Total waktu      : {time.time()-t_global:.1f}s")
print(f"{'='*80}")

class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        if isinstance(obj, (np.int32, np.int64)):
            return int(obj)
        if isinstance(obj, (np.float32, np.float64)):
            return float(obj)
        return super(NumpyEncoder, self).default(obj)

summary = {
    'version': 'v87',
    'strategy': 'Weighted_Ensemble + CrossModal(MFCC+W2V) + OOF_Stacking',
    'best_single': {'model': best_single['model'], 'scenario': best_single['scenario'],
                    'n': best_single['best_n'], 'f1_oof': best_single['test_f1_oof']},
    'best_weighted_ens': best_weighted_info,
    'best_stacking': best_stack_info,
    'overall_best': round(overall_best_v87, 4),
    'target_075': bool(overall_best_v87 >= 0.75),
    'v86_ref': 0.7494,
}
json.dump(summary, open(os.path.join(RESULTS_DIR,"metrics","v87_summary.json"),'w'), indent=2, cls=NumpyEncoder)
print(f"  Summary saved: {RESULTS_DIR}/metrics/v87_summary.json")


                            TABEL RINGKASAN v87 — S1-S4 × 5 Models (OOF Threshold)                            
  Skenario               Model        n      CV F1  Test(oof)  Test(sw)    AUC
  S3_Wav2Vec             XGB          35    0.5889     0.7494    0.7494 0.7600
  S3_Wav2Vec             SVM_lin      30    0.6416     0.7000    0.7980 0.7400
  S2_MFCC                RF           10    0.7097     0.6703    0.7980 0.7800
  S2_MFCC                XGB          10    0.6906     0.6011    0.8465 0.7900
  S4_Fusion              LR           10    0.6640     0.6011    0.6970 0.6000
  S2_MFCC                SVM_rbf      10    0.7448     0.5960    0.6491 0.7100
  S3_Wav2Vec             LR           30    0.6768     0.5960    0.7917 0.7600
  S4_Fusion              SVM_lin      25    0.6163     0.5960    0.5960 0.5100
  S3_Wav2Vec             RF           30    0.6487     0.5833    0.8000 0.7500
  S4_Fusion              XGB          10    0.6258     0.5833    0.6491 0.5600
  S4_Fusion        